In [ ]:
import os
from datetime import datetime

import pandas as pd
import seaborn as sns
import torch
from datasets import load_from_disk
from torch.nn.utils import get_total_norm, clip_grad_norm_

from torch.utils.tensorboard import SummaryWriter
from torchmetrics.text.rouge import ROUGEScore
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from tqdm import tqdm

In [ ]:
DEVICE = torch.device("cuda:0")

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({"pad_token": "<|endoftext|>"})
tokenizer.add_eos_token = True

In [ ]:
SFT_CHECKPOINT_PATH = '/home/logsumexp/workspace/rl-summarizer/artifacts/sft_Mar31_07-34-36_lr=3e-05_weight_decay=0_clip_grad_val=inf'

In [ ]:
reward_model = GPT2ForSequenceClassification.from_pretrained(SFT_CHECKPOINT_PATH)
reward_model.config.pad_token_id = tokenizer.pad_token_id
# model.set_attn_implementation("eager")
# model.config.output_attentions = True
reward_model.to(DEVICE)
# if DEVICE.type == "cuda":
#     model.set_attn_implementation("flash_attention_2")

# Data preparation

In [ ]:
# comparison_dataset_train = load_from_disk("datasets/comparison/train")
# comparison_dataset_val = load_from_disk("datasets/comparison/val")

In [ ]:
def get_winner_loser(x):
    post = "You are given a text. Your task is to write a concise summary for this text. Text: " + x['info']['post'] + " Summary: "
    summary_0 = x['summaries'][0]['text']
    summary_1 = x['summaries'][1]['text']

    if x['choice'] == 1:
        x['winner'] = post + summary_1
        x['loser'] = post + summary_0
    else:
        x['winner'] = post + summary_0
        x['loser'] = post + summary_1
    return x

In [ ]:
# comparison_dataset_train = comparison_dataset_train.map(get_winner_loser)
# comparison_dataset_val = comparison_dataset_val.filter(lambda x: x['info'] is not None and x['info']['post'] is not None).map(get_winner_loser)

In [ ]:
# comparison_dataset_train.save_to_disk("datasets/comparison/train_preprocessed")
# comparison_dataset_val.save_to_disk("datasets/comparison/val_preprocessed")

In [ ]:
comparison_dataset_train = load_from_disk("datasets/comparison/train_preprocessed")
comparison_dataset_val = load_from_disk("datasets/comparison/val_preprocessed")

In [ ]:
len(comparison_dataset_train)

In [ ]:
len(comparison_dataset_val)

In [ ]:
def collate_fn(batch):
    winner_inputs = tokenizer(
        [_['winner'] for _ in batch], padding=True, truncation=True,
        return_tensors="pt", add_special_tokens=True, padding_side="left",
    )

    for k, v in winner_inputs.items():
        winner_inputs[k] = v.to(DEVICE)

    loser_inputs = tokenizer(
        [_['loser'] for _ in batch], padding=True, truncation=True,
        return_tensors="pt", add_special_tokens=True, padding_side="left",
    )

    for k, v in loser_inputs.items():
        loser_inputs[k] = v.to(DEVICE)

    return winner_inputs, loser_inputs

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0

In [ ]:
rm_train_dl = torch.utils.data.DataLoader(comparison_dataset_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
rm_val_dl = torch.utils.data.DataLoader(comparison_dataset_val, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)

In [ ]:
sample_winner_inputs, sample_loser_inputs = iter(rm_train_dl).__next__()

In [ ]:
reward_model.eval()

with torch.no_grad():
    winner_logits = reward_model(**sample_winner_inputs).logits[:, 1]
    loser_logits = reward_model(**sample_loser_inputs).logits[:, 1]

In [ ]:
logsigmoid = torch.nn.LogSigmoid()
loss = -logsigmoid(winner_logits - loser_logits).mean()
loss

In [ ]:
logsigmoid = torch.nn.LogSigmoid()

def compute_loss(model, batch, do_not_need_grad=False):
    sample_winner_inputs, sample_loser_inputs = batch

    with torch.autograd.grad_mode.inference_mode(mode=do_not_need_grad):
        winner_logits  = model(**sample_winner_inputs).logits[:, 1]
        loser_logits  = model(**sample_loser_inputs).logits[:, 1]

    return -logsigmoid(winner_logits - loser_logits).mean()

In [ ]:
compute_loss(reward_model, (sample_winner_inputs, sample_loser_inputs), do_not_need_grad=True)